# Caso 9 — Análise de Erros

Investigamos dois tipos de erro nos rankings produzidos pelo Modelo
Vetorial e pelo BM25, na configuração padrão `stopwords_stemming`.

A e B são documentos não relevantes que aparecem nas primeiras posições
do ranking, com pelo menos 2 exemplos. C é um documento relevante que
não aparece no Top 10, ficando na verdade muito abaixo dele.

Os casos A e C usam a mesma consulta, 31, deliberadamente, pois ela
ilustra simultaneamente os dois tipos de erro de forma conectada.


In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pickle
import pandas as pd
from collections import Counter
from IPython.display import display

from src.cranfield_data import load_cranfield
from src.pre_processing import load_preprocessed

df_docs, df_queries, df_qrels = load_cranfield()
preprocessed = load_preprocessed(project_root / "data" / "processed" / "preprocessed_cranfield.pkl")
doc_tokens = preprocessed["stopwords_stemming"]["docs"]
query_tokens_list = preprocessed["stopwords_stemming"]["queries"]
doc_ids = df_docs["doc_id"].tolist()
query_ids = df_queries["query_id"].tolist()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
doc_text = dict(zip(df_docs.doc_id, df_docs.text.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

RESULTS_DIR = project_root / "data" / "processed"
with open(RESULTS_DIR / "vsm_rankings.pkl", "rb") as f:
    vsm_rankings = pickle.load(f)
with open(RESULTS_DIR / "bm25_rankings.pkl", "rb") as f:
    bm25_rankings = pickle.load(f)


def term_overlap(qid, doc_id):
    qtoks = query_tokens_list[query_ids.index(qid)]
    dtoks = doc_tokens[doc_ids.index(doc_id)]
    counts = Counter(dtoks)
    shared = {t: counts[t] for t in set(qtoks) if counts[t] > 0}
    missing = sorted(t for t in set(qtoks) if counts[t] == 0)
    return qtoks, dtoks, shared, missing

## Caso A — Falso positivo por sobreposição de técnica experimental

A consulta 31 pergunta especificamente sobre o **tamanho** de placas de
extremidade ("end plates") para simular escoamento bidimensional. O
documento 751, julgado **não relevante**, ainda assim ocupa o **1º
lugar** em ambos os modelos.


In [ ]:
# Caso A: documento NÃO relevante no topo do ranking
qid, doc_id = "31", "751"
grade = df_qrels[(df_qrels.query_id == qid) & (df_qrels.doc_id == doc_id)]["relevance"].iloc[0]
rank_bm25 = bm25_rankings[qid].index(doc_id) + 1
rank_vsm = vsm_rankings[qid].index(doc_id) + 1
print(f"Consulta {qid}: {query_text[qid]}")
print(f"Documento {doc_id} (grau={grade}, NÃO RELEVANTE): {doc_title[doc_id]}")
print(f"Posição no ranking -> BM25: {rank_bm25}º | Modelo Vetorial: {rank_vsm}º")
print()

qtoks, dtoks, shared, missing = term_overlap(qid, doc_id)
print(f"Tokens da consulta ({len(qtoks)}): {qtoks}")
print(f"Termos em comum com o documento: {shared}")
print(f"Termos da consulta ausentes no documento: {missing}")
print()
print("Texto completo do documento:")
print(" ", doc_text[doc_id])

Consulta 31: what size of end plate can be safely used to simulate two-dimensional flow conditions over a bluff cylindrical body of finite aspect ratio .
Documento 751 (grau=-1, NÃO RELEVANTE): a note on the use of end plates to prevent three dimensional flow at the ends of bluff cylinders .
Posição no ranking -> BM25: 1º | Modelo Vetorial: 1º

Tokens da consulta (14): ['size', 'end', 'plate', 'safe', 'use', 'simul', 'flow', 'condit', 'bluff', 'cylindr', 'bodi', 'finit', 'aspect', 'ratio']
Termos em comum com o documento: {'end': 6, 'flow': 3, 'use': 2, 'plate': 3, 'cylindr': 1, 'bluff': 2}
Termos da consulta ausentes no documento: ['aspect', 'bodi', 'condit', 'finit', 'ratio', 'safe', 'simul', 'size']

Texto completo do documento:
  a note on the use of end plates to prevent three dimensional flow at the ends of bluff cylinders .   the results are given of some observations of the effects of end plates on the three-dimensional separated flow at the ends of cylindrical models .  while 

Por quê. O documento 751 compartilha 6 dos 16 termos da consulta,
incluindo os termos centrais da técnica experimental, `end`, `plate`,
`flow`, `dimension`, `bluff` e `cylindr`, uma sobreposição lexical
genuína, não coincidência. O problema é de especificidade da pergunta.
O documento trata de como evitar o efeito tridimensional nas
extremidades de um cilindro, ou seja, da técnica de forma geral, mas não
aborda a pergunta específica da consulta, que é qual tamanho de placa é
suficiente. Nenhum termo relacionado a dimensionamento, como `size`,
`aspect`, `ratio` ou `finit`, aparece no documento. Um modelo lexical
não consegue distinguir descrever a técnica de responder à pergunta
quantitativa específica sobre a técnica, pois ambos produzem o mesmo
tipo de sobreposição de palavras-chave.


## Caso B — Falso positivo por mesmo método, domínio diferente

Um segundo exemplo, com um mecanismo relacionado, porém distinto.


In [ ]:
# Caso B: um segundo documento NÃO relevante no topo do ranking
qid, doc_id = "173", "532"
grade = df_qrels[(df_qrels.query_id == qid) & (df_qrels.doc_id == doc_id)]["relevance"].iloc[0]
rank_bm25 = bm25_rankings[qid].index(doc_id) + 1
rank_vsm = vsm_rankings[qid].index(doc_id) + 1
print(f"Consulta {qid}: {query_text[qid]}")
print(f"Documento {doc_id} (grau={grade}, NÃO RELEVANTE): {doc_title[doc_id]}")
print(f"Posição no ranking -> BM25: {rank_bm25}º | Modelo Vetorial: {rank_vsm}º")
print()

qtoks, dtoks, shared, missing = term_overlap(qid, doc_id)
print(f"Tokens da consulta ({len(qtoks)}): {qtoks}")
print(f"Termos em comum com o documento: {shared}")
print(f"Termos da consulta ausentes no documento: {missing}")

Consulta 173: references on lyapunov's method on the stability of linear differential equations with periodic coefficients .
Documento 532 (grau=-1, NÃO RELEVANTE): pitch-yaw stability of a missile oscillating in roll via the second method of lyapunov .
Posição no ranking -> BM25: 3º | Modelo Vetorial: 1º

Tokens da consulta (9): ['refer', 'lyapunov', 'method', 'stabil', 'linear', 'differenti', 'equat', 'period', 'coeffici']
Termos em comum com o documento: {'stabil': 4, 'method': 2, 'lyapunov': 4}
Termos da consulta ausentes no documento: ['coeffici', 'differenti', 'equat', 'linear', 'period', 'refer']


Por quê. A consulta 173 busca referências teóricas sobre o método de
Lyapunov aplicado a equações diferenciais lineares com coeficientes
periódicos, um tópico de matemática e teoria de controle. O documento
532 também usa a expressão o segundo método de Lyapunov e discute
estabilidade, mas aplicado a um problema de engenharia bem específico,
a oscilação de rolagem de um míssil. Os termos mais raros e informativos
da consulta, `lyapunov`, de idf alto, `stabil` e `method`, aparecem
repetidas vezes no documento, e por isso ele pontua tão bem. Mas os
termos que sinalizariam o tratamento teórico geral que a consulta pede,
`linear`, `equat`, `differenti`, `period` e `coeffici`, estão todos
ausentes. É o padrão de aplicação específica encontrada quando se busca
teoria geral, um erro diferente do Caso A, em que era a mesma técnica
com uma pergunta diferente sobre ela. Aqui é o mesmo método citado pelo
nome, mas com nível de generalidade diferente.


## Caso C — Documento relevante "invisível" para os dois modelos

Na mesma consulta 31, o único documento julgado relevante (grau 4) é o
documento 776 e ele está longe de qualquer posição útil do ranking.


In [ ]:
# Caso C: documento RELEVANTE que não aparece no Top-10 (nem perto dele)
qid, doc_id = "31", "776"
grade = df_qrels[(df_qrels.query_id == qid) & (df_qrels.doc_id == doc_id)]["relevance"].iloc[0]
rank_bm25 = bm25_rankings[qid].index(doc_id) + 1
rank_vsm = vsm_rankings[qid].index(doc_id) + 1
print(f"Consulta {qid}: {query_text[qid]}")
print(f"Documento {doc_id} (grau={grade}, RELEVANTE): {doc_title[doc_id]}")
print(f"Posição no ranking -> BM25: {rank_bm25}º de {len(bm25_rankings[qid])} "
      f"| Modelo Vetorial: {rank_vsm}º de {len(vsm_rankings[qid])}")
print()

qtoks, dtoks, shared, missing = term_overlap(qid, doc_id)
print(f"Tokens da consulta ({len(qtoks)}): {qtoks}")
print(f"Termos em comum com o documento: {shared}")
print(f"Termos da consulta ausentes no documento: {missing}")
print()
print("Texto completo do documento:")
print(" ", doc_text[doc_id])

Consulta 31: what size of end plate can be safely used to simulate two-dimensional flow conditions over a bluff cylindrical body of finite aspect ratio .
Documento 776 (grau=4, RELEVANTE): force measurements on square and dodecagonal sectional cylinders at high reynolds numbers .
Posição no ranking -> BM25: 1301º de 1400 | Modelo Vetorial: 1301º de 1400

Tokens da consulta (14): ['size', 'end', 'plate', 'safe', 'use', 'simul', 'flow', 'condit', 'bluff', 'cylindr', 'bodi', 'finit', 'aspect', 'ratio']
Termos em comum com o documento: {}
Termos da consulta ausentes no documento: ['aspect', 'bluff', 'bodi', 'condit', 'cylindr', 'end', 'finit', 'flow', 'plate', 'ratio', 'safe', 'simul', 'size', 'use']

Texto completo do documento:
  force measurements on square and dodecagonal sectional cylinders at high reynolds numbers .   results are given of measurements in the compressed air tunnel of the forces on two cylinders, one of square cross-section and the other dodecagonal .  the tests were c

**Por quê**: a sobreposição de termos entre a consulta e o documento 776
é de exatamente **1 termo** (`two`) — e mesmo esse é uma coincidência
semântica: na consulta, "two" vem de "two-dimensional flow"; no
documento, vem de "forces on **two** cylinders" (uma contagem, sem
relação com dimensionalidade). Ou seja, na prática, a sobreposição real
de conteúdo é **zero**. O documento relata medições de força em
cilindros de seção quadrada e dodecagonal — dados experimentais que,
segundo o julgamento humano, informam a pergunta sobre dimensionamento de
placas de extremidade (provavelmente pela relação entre as duas
montagens experimentais descrita no corpo do artigo original), mas essa
relação **não está no vocabulário de superfície** do resumo indexado.


Há ainda um segundo problema, mais sutil e verificável: o documento usa
a palavra "cylinders", e a consulta usa "cylindrical" — duas formas da
mesma raiz. Elas deveriam ser unificadas pelo stemming, mas não são:


In [ ]:
# Investigando mais a fundo: por que "cylindrical" (da consulta) e
# "cylinders" (do documento 776) não se encontram, mesmo os dois vindo
# claramente da mesma raiz semântica?
from nltk.stem import PorterStemmer
ps = PorterStemmer()
for palavra in ["cylindrical", "cylinder", "cylinders"]:
    print(f"  stem('{palavra}') = '{ps.stem(palavra)}'")

  stem('cylindrical') = 'cylindr'
  stem('cylinder') = 'cylind'
  stem('cylinders') = 'cylind'


O Porter Stemmer reduz `cylindrical` a `cylindr`, mas `cylinder` e
`cylinders` a `cylind`, gerando stems diferentes para a mesma família de
palavras. É uma inconsistência conhecida desse algoritmo, que aplica
regras de sufixo sem nenhum dicionário ou lematização real. Mesmo que o
restante do vocabulário coincidisse, esse termo específico não
contribuiria para a similaridade em nenhum dos dois modelos. Isso mostra
que a análise de erros de recuperação não pode parar em dizer que os
textos são diferentes. Vale a pena checar também se a etapa de
pré-processamento está introduzindo falhas de correspondência que um
dicionário de sinônimos ou um lematizador, em vez de um stemmer
puramente baseado em regras, poderiam evitar.


## Síntese

Os três casos ilustram três mecanismos de erro distintos, todos
originados da mesma limitação fundamental, que é a correspondência
lexical em vez de semântica.

No Caso A há sobreposição genuína de vocabulário da técnica, mas
ausência do vocabulário que sinalizaria a pergunta específica sobre
dimensionamento sendo respondida.

No Caso B há sobreposição genuína dos termos mais raros e informativos,
como o nome do método, mas confusão entre o nível de generalidade, ou
seja, teoria geral contra aplicação específica.

No Caso C há ausência quase total de vocabulário compartilhado,
agravada por uma inconsistência de stemming que impede a unificação de
uma variação morfológica que, sozinha, já ajudaria pouco mas não é nula.

Nenhum dos três seria resolvido apenas trocando o modelo de ranking,
Vetorial por BM25 ou vice versa, pois os dois modelos falham exatamente
da mesma forma nesses casos, já que o problema está na representação,
do tipo bag of words lexical, e não no cálculo do score. Técnicas como
expansão de consultas, tesauros de sinônimos ou representações
semânticas dos termos, como embeddings, atacariam diretamente esse tipo
de erro, mas estão fora do escopo dos modelos clássicos implementados
neste projeto.
